# PB02 — Confronto Segnale: img vs read vs rest

**Obiettivo**: capire le differenze strutturali tra le tre condizioni per la stessa parola e soggetto.

**Analisi**:
- ERP medio per condizione
- Spettro di potenza per condizione
- Topomap per banda (alpha, beta)
- Test statistico Wilcoxon img vs read vs rest

In [ ]:
from pathlib import Path

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists()),
    Path().resolve()
)

DATA_ROOT = project_root / 'data' / 'raw_csv' / 'training_set'
SFREQ = 256
N_CHAN = 61
N_SAMP = 384
TIMES = [i / SFREQ for i in range(N_SAMP)]  # 0 → 1.5s

# Soggetti da analizzare (es. campione rappresentativo)
SUBJ_IDS = [0, 1, 5, 10, 20]

print(f'DATA_ROOT: {DATA_ROOT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy import signal, stats
from tqdm.auto import tqdm

mne.set_log_level('WARNING')
print('Import OK')

In [ ]:
# ============================================================
# CARICAMENTO DATI
# ============================================================

def load_condition(subj_id: int, condition: str, data_root: Path, max_words: int = None) -> np.ndarray:
    """
    condition: 'img', 'read', 'rest'
    Ritorna (n_trials, 61, 384)
    """
    epochs = []
    for sess_dir in sorted(data_root.glob(f'P{subj_id:03d}_S*')):
        if condition in ('img', 'read'):
            pattern = f'*_{condition}.csv'
        else:
            pattern = 'riposo_*.csv'
        for f in sorted(sess_dir.glob(pattern)):
            x = pd.read_csv(f, header=None).values.astype(np.float32)
            if x.shape == (N_CHAN, N_SAMP):
                epochs.append(x)
            if max_words and len(epochs) >= max_words:
                break
    return np.array(epochs) if epochs else np.empty((0, N_CHAN, N_SAMP), dtype=np.float32)


# Test
sid = SUBJ_IDS[0]
for cond in ('img', 'read', 'rest'):
    ep = load_condition(sid, cond, DATA_ROOT)
    print(f'P{sid:03d} {cond}: {ep.shape}')

In [ ]:
# ============================================================
# ERP MEDIO PER CONDIZIONE (media su trial e canali)
# ============================================================

fig, axes = plt.subplots(1, len(SUBJ_IDS), figsize=(4 * len(SUBJ_IDS), 4), sharey=True)

for ax, sid in zip(axes, SUBJ_IDS):
    for cond, color in [('img', 'steelblue'), ('read', 'darkorange'), ('rest', 'seagreen')]:
        ep = load_condition(sid, cond, DATA_ROOT)
        if len(ep) == 0:
            continue
        erp = ep.mean(axis=(0, 1))  # media su trial e canali → (384,)
        ax.plot(TIMES, erp, label=cond, color=color, alpha=0.8)
    ax.set_title(f'P{sid:03d}')
    ax.set_xlabel('Tempo (s)')
    ax.axvline(0, color='k', linestyle='--', linewidth=0.5)

axes[0].set_ylabel('Ampiezza (µV)')
axes[0].legend()
fig.suptitle('ERP medio per condizione')
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB02_erp_comparison.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# SPETTRO DI POTENZA PER CONDIZIONE (media su soggetti)
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

for cond, color in [('img', 'steelblue'), ('read', 'darkorange'), ('rest', 'seagreen')]:
    psds_all = []
    for sid in tqdm(SUBJ_IDS, desc=cond, leave=False):
        ep = load_condition(sid, cond, DATA_ROOT)
        if len(ep) == 0:
            continue
        freqs, psd = signal.welch(ep, fs=SFREQ, nperseg=SFREQ, axis=-1)  # (n, ch, f)
        psds_all.append(psd.mean(axis=(0, 1)))  # media su trial e canali → (f,)
    if psds_all:
        mean_psd = np.mean(psds_all, axis=0)
        std_psd  = np.std(psds_all, axis=0)
        ax.semilogy(freqs, mean_psd, label=cond, color=color)
        ax.fill_between(freqs, mean_psd - std_psd, mean_psd + std_psd, alpha=0.2, color=color)

for band, freq in [('delta', 4), ('theta', 8), ('alpha', 12), ('beta', 30)]:
    ax.axvline(freq, color='gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(freq, ax.get_ylim()[1], band, fontsize=7, color='gray', ha='center')

ax.set_xlim(1, 45)
ax.set_xlabel('Frequenza (Hz)')
ax.set_ylabel('PSD (µV²/Hz)')
ax.set_title('Spettro di potenza per condizione')
ax.legend()
plt.tight_layout()
plt.savefig(project_root / 'figures' / 'PB02_psd_comparison.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# TEST STATISTICO: img vs read vs rest (Wilcoxon su varianza)
# Per ogni soggetto, compara varianza media delle tre condizioni
# ============================================================

results = []
for sid in tqdm(SUBJ_IDS, desc='Soggetti'):
    row = {'subj': sid}
    for cond in ('img', 'read', 'rest'):
        ep = load_condition(sid, cond, DATA_ROOT)
        if len(ep) > 0:
            row[f'var_{cond}'] = float(ep.var(axis=-1).mean())
            row[f'n_{cond}']   = len(ep)
    results.append(row)

df = pd.DataFrame(results)
print(df[['subj', 'var_img', 'var_read', 'var_rest', 'n_img', 'n_read', 'n_rest']].to_string())

# Wilcoxon img vs read
stat, p = stats.wilcoxon(df['var_img'].dropna(), df['var_read'].dropna())
print(f'\nWilcoxon img vs read: stat={stat:.2f}  p={p:.4f}')
stat, p = stats.wilcoxon(df['var_img'].dropna(), df['var_rest'].dropna())
print(f'Wilcoxon img vs rest: stat={stat:.2f}  p={p:.4f}')